In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x10c7191d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10c87d0d0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie's rating out of 10")

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x10c7191d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10c87d0d0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': "The movie's rating out of 10", 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': 

In [4]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user asks for "details about the movie Inception". This is a straightforward request for comprehensive information about the film.\n\n2.  **Identify Key Information Needed**:\n   - Title: Inception\n   - Release Year: 2010\n   - Director: Christopher Nolan\n   - Writers: Christopher Nolan\n   - Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Cillian Murphy, Marion Cotillard, Michael Caine, etc.\n   - Genre: Sci-Fi, Action, Thriller, Heist\n   - Plot Summary: Core premise, main characters, mission, dream layers, climax\n   - Themes: Reality vs. dreams, guilt, subconscious, time perception, architecture of dreams\n   - Critical Reception: Box office, reviews, awards/nominations\n   - Cinematic Techniques: Practical effects, IMAX, sound design, score (Hans Zimmer)\n   - Notable Elements: Spinning top ending, dream-within-a-dream structure, limbo\n   - Lega

In [8]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie "Inception".\n2.  **Identify Required Information:** I need to provide details about the movie. The available tool is `Movie`, which requires: `title`, `year`, `director`, and `rating`.\n3.  **Check Tool Parameters:**\n   - `title`: "Inception"\n   - `year`: 2010\n   - `director`: Christopher Nolan\n   - `rating`: I need to provide a rating out of 10. Inception is highly rated, typically around 8.8/10 on IMDb. I\'ll use 8.8.\n4.  **Construct Tool Call:** I have all the required parameters. I\'ll call the `Movie` function with these values.\n5.  **Execute Tool Call:** (Mental simulation)\n   ```json\n   {\n     "name": "Movie",\n     "parameters": {\n       "title": "Inception",\n       "year": 2010,\n       "director": "Christopher Nolan",\n       "rating": 8.8\n     }\n   }\n   ```\n6.  **Generate Response:** T

In [10]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

In [11]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}